In [ ]:
import pandas as pd

In [ ]:
# Real dataset: ORCAS-I (built from real, anonymized Bing search queries)
# Source: TU Wien Research Data, CC-BY 4.0 license
gold_url = "https://researchdata.tuwien.ac.at/records/pp7xz-n9a06/files/ORCAS-I-gold.tsv?download=1"
sample_url = "https://researchdata.tuwien.ac.at/records/pp7xz-n9a06/files/ORCAS-I-2M.tsv?download=1"

In [ ]:
# Step 1a: download the small "gold" test file first (only ~100 KB, human-checked labels)
gold_df = pd.read_csv(gold_url, sep="\t")
print("Gold test set shape:", gold_df.shape)
print(gold_df.head())
print(gold_df["label_manual"].value_counts())

Gold test set shape: (1000, 6)
        qid                             query       did  \
0   7916625                        best reads   D889100   
1   7737755                          tamerind   D586723   
2   4598644                        show mi ip  D3188590   
3  11008126        do carpenter ants eat wood  D1593016   
4   7737808  rheumatoid arthritis in children  D2557045   

                                                 url  label_manual data_split  
0                       http://thegreatestbooks.org/       Abstain       test  
1             https://en.wikipedia.org/wiki/Tamarind       Factual       test  
2                                 http://showip.net/  Instrumental       test  
3          https://doyourownpestcontrol.com/carp.htm       Factual       test  
4  https://www.webmd.com/rheumatoid-arthritis/und...       Abstain       test  
label_manual
Abstain          364
Factual          363
Navigational     171
Instrumental      59
Transactional     43
Name: count, dty

In [ ]:
# Step 2: load the big file, but only the columns we actually need (saves memory + time)
cols_needed = ["qid", "query", "level_1", "data_split"]
big_df = pd.read_csv(sample_url, sep="\t", usecols=cols_needed)

print("Full file shape:", big_df.shape)
print(big_df["level_1"].value_counts())

Full file shape: (2000000, 4)
level_1
Informational    1627056
Navigational      289610
Transactional      83334
Name: count, dtype: int64


In [ ]:
# Step 3: take a balanced sample - equal amount from each category
# We don't need millions of rows. A few thousand per category is plenty
# for a simple TF-IDF + Logistic Regression model to learn well and train fast.

SAMPLES_PER_CLASS = 4000   # small enough to train in seconds, big enough to learn patterns

balanced_parts = []
for label in ["Informational", "Navigational", "Transactional"]:
    part = big_df[big_df["level_1"] == label].sample(
        n=SAMPLES_PER_CLASS, random_state=42
    )
    balanced_parts.append(part)

train_sample = pd.concat(balanced_parts).sample(frac=1, random_state=42)  # shuffle rows
train_sample = train_sample.reset_index(drop=True)

print("Balanced sample shape:", train_sample.shape)
print(train_sample["level_1"].value_counts())

# Save it, so we don't need to re-download/re-sample the big file every time
train_sample.to_csv("orcas_train_sample.csv", index=False)
gold_df.to_csv("orcas_gold_test.csv", index=False)

Balanced sample shape: (12000, 4)
level_1
Informational    4000
Navigational     4000
Transactional    4000
Name: count, dtype: int64


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

train_sample = pd.read_csv("orcas_train_sample.csv")

# Step 4a: light cleaning ONLY - lowercase and remove extra spaces.
# We are NOT removing punctuation or numbers here. More on why below.
train_sample["query_clean"] = train_sample["query"].str.lower().str.strip()

# Step 4b: split our sample into a training part and a small internal check part
X_train, X_val, y_train, y_val = train_test_split(
    train_sample["query_clean"],
    train_sample["level_1"],
    test_size=0.2,
    random_state=42,
    stratify=train_sample["level_1"],  # keeps equal mix of classes in both parts
)

# Step 4c: turn text into numbers
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)

# Step 4d: train the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# Step 4e: check how it's doing (on our OWN internal check data - NOT the gold file yet)
preds = model.predict(X_val_vec)
print(classification_report(y_val, preds))

               precision    recall  f1-score   support

Informational       0.67      0.83      0.74       800
 Navigational       0.78      0.61      0.68       800
Transactional       0.99      0.99      0.99       800

     accuracy                           0.81      2400
    macro avg       0.82      0.81      0.81      2400
 weighted avg       0.82      0.81      0.81      2400



In [ ]:
# Step 5: Testing Hypothesis H1
# H1 says: simple, well-known single-name queries should get HIGH confidence (>90%) as Navigational

test_queries = [
    "youtube", "gmail", "facebook", "amazon", "netflix",
    "instagram", "whatsapp web", "hdfc netbanking", "flipkart", "linkedin"
]

test_clean = [q.lower().strip() for q in test_queries]
test_vec = vectorizer.transform(test_clean)

probs = model.predict_proba(test_vec)
classes = model.classes_

result = pd.DataFrame(probs, columns=classes)
result.insert(0, "query", test_queries)
result["predicted"] = model.predict(test_vec)
print(result)

             query  Informational  Navigational  Transactional      predicted
0          youtube       0.400241      0.420395       0.179364   Navigational
1            gmail       0.491262      0.447714       0.061024  Informational
2         facebook       0.147141      0.796826       0.056034   Navigational
3           amazon       0.356461      0.534354       0.109185   Navigational
4          netflix       0.574447      0.339744       0.085810  Informational
5        instagram       0.477598      0.423262       0.099140  Informational
6     whatsapp web       0.116277      0.814287       0.069436   Navigational
7  hdfc netbanking       0.377472      0.549373       0.073155   Navigational
8         flipkart       0.476720      0.453945       0.069335  Informational
9         linkedin       0.258483      0.687682       0.053835   Navigational


In [ ]:
# Step 6: check if these words actually exist in the model's vocabulary,
# and how many times they appeared in training data

vocab = vectorizer.vocabulary_

for word in ["youtube", "gmail", "facebook", "amazon", "netflix", "instagram", "flipkart", "linkedin"]:
    in_vocab = word in vocab
    count_in_training = train_sample["query_clean"].str.contains(rf"\b{word}\b", regex=True).sum()
    print(f"{word:12s} | in vocabulary: {in_vocab} | appears in training queries: {count_in_training} times")

youtube      | in vocabulary: True | appears in training queries: 21 times
gmail        | in vocabulary: True | appears in training queries: 13 times
facebook     | in vocabulary: True | appears in training queries: 30 times
amazon       | in vocabulary: True | appears in training queries: 57 times
netflix      | in vocabulary: True | appears in training queries: 18 times
instagram    | in vocabulary: True | appears in training queries: 7 times
flipkart     | in vocabulary: False | appears in training queries: 0 times
linkedin     | in vocabulary: True | appears in training queries: 3 times


In [ ]:
# Step 7: check WHICH labels these words are actually paired with in training data

for word in ["amazon", "facebook", "netflix", "gmail", "instagram"]:
    matches = train_sample[train_sample["query_clean"].str.contains(rf"\b{word}\b", regex=True)]
    print(f"\n--- {word} ({len(matches)} training queries) ---")
    print(matches["level_1"].value_counts())
    print(matches["query_clean"].head(5).tolist())


--- amazon (57 training queries) ---
level_1
Transactional    29
Navigational     18
Informational    10
Name: count, dtype: int64
['amazon phone number uk 0800', 'amazon prime tv app', 'free gift cards amazon', 'download amazon prime', 'did amazon buy whole foods']

--- facebook (30 training queries) ---
level_1
Navigational     16
Transactional    12
Informational     2
Name: count, dtype: int64
['my facebook account has been hacked now what', 'recover disabled facebook account', 'facebook payment customer service number', 'messenger app for facebook', 'facebook login sverige']

--- netflix (18 training queries) ---
level_1
Transactional    11
Informational     5
Navigational      2
Name: count, dtype: int64
['get netflix free', 'watch netflix shows online free', 'netflix 1 month free', 'netflix movies free download', 'programs on netflix']

--- gmail (13 training queries) ---
level_1
Navigational     9
Transactional    2
Informational    2
Name: count, dtype: int64
['gmail password

In [ ]:
# Step 8: check EXACT single-word brand queries only (not "contains")
# This is the correct, precise way to actually test H1

for word in ["amazon", "facebook", "netflix", "gmail", "instagram", "youtube"]:
    exact_matches = train_sample[train_sample["query_clean"] == word]
    print(f"\n--- exactly '{word}' ({len(exact_matches)} training queries) ---")
    if len(exact_matches) > 0:
        print(exact_matches["level_1"].value_counts())
    else:
        print("No exact single-word match found in our training sample.")


--- exactly 'amazon' (0 training queries) ---
No exact single-word match found in our training sample.

--- exactly 'facebook' (1 training queries) ---
level_1
Navigational    1
Name: count, dtype: int64

--- exactly 'netflix' (0 training queries) ---
No exact single-word match found in our training sample.

--- exactly 'gmail' (0 training queries) ---
No exact single-word match found in our training sample.

--- exactly 'instagram' (1 training queries) ---
level_1
Navigational    1
Name: count, dtype: int64

--- exactly 'youtube' (0 training queries) ---
No exact single-word match found in our training sample.


In [ ]:
!pip install rapidfuzz -q

from rapidfuzz import process, fuzz

# A small, honestly self-built list of well-known real site/brand names.
# This is a real, disclosed list - not scraped from anywhere, just common
# well-known names (a mix of global + India-relevant sites, since that's
# our context). This kind of fixed list is a real, standard NLP technique
# called a "gazetteer" - it's not a workaround, it's normal practice.
KNOWN_SITES = [
    "youtube", "gmail", "facebook", "instagram", "amazon", "netflix",
    "flipkart", "linkedin", "whatsapp", "twitter", "snapchat", "pinterest",
    "reddit", "wikipedia", "github", "spotify", "zoom", "outlook", "yahoo",
    "myntra", "swiggy", "zomato", "uber", "ola", "irctc", "paytm",
    "hdfc netbanking", "icici bank", "sbi online", "google", "microsoft",
]

def check_known_site(query, threshold=90):
    query = query.lower().strip()
    match, score, _ = process.extractOne(query, KNOWN_SITES, scorer=fuzz.ratio)
    if score >= threshold:
        return True, match, score
    return False, None, score

# quick test - includes correct spellings AND typos together
test_queries = ["youtube", "goggle", "yuotube", "gmial", "netfl!x", "random question about weather"]

for q in test_queries:
    is_known, matched_to, score = check_known_site(q)
    print(f"{q:30s} -> known site: {is_known} | matched: {matched_to} | score: {score:.0f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 32.1 MB/s eta 0:00:00
youtube                        -> known site: True | matched: youtube | score: 100
goggle                         -> known site: False | matched: None | score: 83
yuotube                        -> known site: False | matched: None | score: 86
gmial                          -> known site: False | matched: None | score: 80
netfl!x                        -> known site: False | matched: None | score: 86
random question about weather  -> known site: False | matched: None | score: 33


In [ ]:
# Step 10: try a lower threshold and re-test, using both typos AND
# a few longer, unrelated queries to make sure we don't break safety

test_queries = [
    "youtube", "goggle", "yuotube", "gmial", "netfl!x",           # typos - should match
    "random question about weather",                              # unrelated - should NOT match
    "amazon prime membership price india",                         # long real query - should NOT match
    "how to recover facebook account",                             # long real query - should NOT match
]

for threshold in [75, 80, 85]:
    print(f"\n=== Testing threshold = {threshold} ===")
    for q in test_queries:
        is_known, matched_to, score = check_known_site(q, threshold=threshold)
        print(f"{q:38s} -> matched: {is_known} | score: {score:.0f}")


=== Testing threshold = 75 ===
youtube                                -> matched: True | score: 100
goggle                                 -> matched: True | score: 83
yuotube                                -> matched: True | score: 86
gmial                                  -> matched: True | score: 80
netfl!x                                -> matched: True | score: 86
random question about weather          -> matched: False | score: 33
amazon prime membership price india    -> matched: False | score: 32
how to recover facebook account        -> matched: False | score: 41

=== Testing threshold = 80 ===
youtube                                -> matched: True | score: 100
goggle                                 -> matched: True | score: 83
yuotube                                -> matched: True | score: 86
gmial                                  -> matched: True | score: 80
netfl!x                                -> matched: True | score: 86
random question about weather          -> match

In [ ]:
# Step 11: the final combined decision function
# Order matters: check the lookup FIRST (fast, certain),
# fall back to the ML model only if it's not a known site

def classify_query(query, model, vectorizer, threshold=80, ml_confidence_cutoff=0.90):
    query_clean = query.lower().strip()

    # Step A: check known-sites lookup first
    is_known, matched_to, score = check_known_site(query_clean, threshold=threshold)
    if is_known:
        return {
            "query": query,
            "predicted": "Navigational",
            "source": "lookup",
            "matched_to": matched_to,
            "confidence": score / 100
        }

    # Step B: not in lookup - fall back to the ML model
    query_vec = vectorizer.transform([query_clean])
    probs = model.predict_proba(query_vec)[0]
    pred_class = model.classes_[probs.argmax()]
    confidence = probs.max()

    # Step C: only trust the model's Navigational guess if confidence is high
    if pred_class == "Navigational" and confidence < ml_confidence_cutoff:
        pred_class = "Informational"  # safe default when unsure - agreed rule

    return {
        "query": query,
        "predicted": pred_class,
        "source": "model",
        "matched_to": None,
        "confidence": confidence
    }

# test it on a mixed set - known sites, typos, and genuinely new queries
final_test = [
    "youtube", "goggle", "flipkart", "gmial",
    "how does compound interest work", "best budget phones 2026",
    "myntra"
]

for q in final_test:
    result = classify_query(q, model, vectorizer)
    print(result)

{'query': 'youtube', 'predicted': 'Navigational', 'source': 'lookup', 'matched_to': 'youtube', 'confidence': 1.0}
{'query': 'goggle', 'predicted': 'Navigational', 'source': 'lookup', 'matched_to': 'google', 'confidence': 0.8333333333333335}
{'query': 'flipkart', 'predicted': 'Navigational', 'source': 'lookup', 'matched_to': 'flipkart', 'confidence': 1.0}
{'query': 'gmial', 'predicted': 'Navigational', 'source': 'lookup', 'matched_to': 'gmail', 'confidence': 0.8}
{'query': 'how does compound interest work', 'predicted': 'Informational', 'source': 'model', 'matched_to': None, 'confidence': np.float64(0.7736330154171792)}
{'query': 'best budget phones 2026', 'predicted': 'Informational', 'source': 'model', 'matched_to': None, 'confidence': np.float64(0.6133581916169281)}
{'query': 'myntra', 'predicted': 'Navigational', 'source': 'lookup', 'matched_to': 'myntra', 'confidence': 1.0}


In [ ]:
import joblib

# Save the trained model, the vectorizer, and our known-sites list as files
joblib.dump(model, "intent_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")
joblib.dump(KNOWN_SITES, "known_sites.pkl")

print("Saved: intent_model.pkl, tfidf_vectorizer.pkl, known_sites.pkl")

# Download them to your computer (Colab will prompt 3 separate downloads)
from google.colab import files
files.download("intent_model.pkl")
files.download("tfidf_vectorizer.pkl")
files.download("known_sites.pkl")

Saved: intent_model.pkl, tfidf_vectorizer.pkl, known_sites.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import sklearn
print(sklearn.__version__)

1.6.1


In [ ]:
print(model)
print(vectorizer)

LogisticRegression(max_iter=1000)
TfidfVectorizer()


In [ ]:
import random
random.seed(42)  # so results are reproducible - same typos every time we run this

def make_typo(word):
    """Make one small, realistic typo: either delete a letter or swap two adjacent ones."""
    word = list(word)
    if len(word) < 3:
        return "".join(word)
    typo_type = random.choice(["delete", "swap"])
    idx = random.randint(0, len(word) - 2)
    if typo_type == "delete":
        del word[idx]
    else:
        word[idx], word[idx+1] = word[idx+1], word[idx]
    return "".join(word)

sample_sites = ["youtube", "gmail", "facebook", "amazon", "netflix",
                "instagram", "flipkart", "linkedin", "myntra", "spotify"]
typo_queries = [(make_typo(site), site) for site in sample_sites]
print("Generated typos:", typo_queries)

# (a) MODEL ONLY - bypass our lookup fix entirely, see how it does raw
print("\n--- Model alone (no fuzzy-matching fix) ---")
model_only_correct = 0
for typo, original in typo_queries:
    query_vec = vectorizer.transform([typo.lower().strip()])
    probs = model.predict_proba(query_vec)[0]
    pred = model.classes_[probs.argmax()]
    correct = (pred == "Navigational")
    model_only_correct += correct
    print(f"'{typo}' (typo of {original}) -> {pred} {'✓' if correct else '✗ WRONG'}")
print(f"\nModel-only accuracy on typos: {model_only_correct}/{len(typo_queries)} = {model_only_correct/len(typo_queries):.0%}")

# (b) FULL HYBRID - our actual fix (lookup + fuzzy matching + model fallback)
print("\n--- Hybrid system (with our fix) ---")
hybrid_correct = 0
for typo, original in typo_queries:
    result = classify_query(typo, model, vectorizer)
    correct = (result["predicted"] == "Navigational")
    hybrid_correct += correct
    print(f"'{typo}' (typo of {original}) -> {result['predicted']} via {result['source']} {'✓' if correct else '✗ WRONG'}")
print(f"\nHybrid accuracy on typos: {hybrid_correct}/{len(typo_queries)} = {hybrid_correct/len(typo_queries):.0%}")

Generated typos: [('outube', 'youtube'), ('gamil', 'gmail'), ('fcebook', 'facebook'), ('amazn', 'amazon'), ('netfix', 'netflix'), ('nistagram', 'instagram'), ('lipkart', 'flipkart'), ('lnkedin', 'linkedin'), ('mynta', 'myntra'), ('spotiy', 'spotify')]

--- Model alone (no fuzzy-matching fix) ---
'outube' (typo of youtube) -> Informational ✗ WRONG
'gamil' (typo of gmail) -> Informational ✗ WRONG
'fcebook' (typo of facebook) -> Informational ✗ WRONG
'amazn' (typo of amazon) -> Informational ✗ WRONG
'netfix' (typo of netflix) -> Informational ✗ WRONG
'nistagram' (typo of instagram) -> Informational ✗ WRONG
'lipkart' (typo of flipkart) -> Informational ✗ WRONG
'lnkedin' (typo of linkedin) -> Informational ✗ WRONG
'mynta' (typo of myntra) -> Informational ✗ WRONG
'spotiy' (typo of spotify) -> Informational ✗ WRONG

Model-only accuracy on typos: 0/10 = 0%

--- Hybrid system (with our fix) ---
'outube' (typo of youtube) -> Navigational via lookup ✓
'gamil' (typo of gmail) -> Navigational via 

In [ ]:
# H3, Part A: does the 0.90 ML confidence cutoff actually prevent wrong guesses?
# Test on queries the model might confidently (but wrongly) call Navigational

risky_queries = [
    "amazon jobs", "netflix subscription cost", "facebook stock price",
    "gmail down today", "instagram algorithm explained",
    "how to delete flipkart account", "linkedin premium worth it"
]

print("--- Effect of the 0.90 confidence cutoff on risky queries ---")
for q in risky_queries:
    query_vec = vectorizer.transform([q.lower().strip()])
    probs = model.predict_proba(query_vec)[0]
    raw_pred = model.classes_[probs.argmax()]
    raw_confidence = probs.max()
    final_result = classify_query(q, model, vectorizer)
    print(f"'{q}'\n  raw model guess: {raw_pred} ({raw_confidence:.2f}) -> after safety rule: {final_result['predicted']} (source: {final_result['source']})\n")

--- Effect of the 0.90 confidence cutoff on risky queries ---
'amazon jobs'
  raw model guess: Navigational (0.48) -> after safety rule: Informational (source: model)

'netflix subscription cost'
  raw model guess: Informational (0.80) -> after safety rule: Informational (source: model)

'facebook stock price'
  raw model guess: Informational (0.48) -> after safety rule: Informational (source: model)

'gmail down today'
  raw model guess: Informational (0.78) -> after safety rule: Informational (source: model)

'instagram algorithm explained'
  raw model guess: Informational (0.55) -> after safety rule: Informational (source: model)

'how to delete flipkart account'
  raw model guess: Informational (0.43) -> after safety rule: Informational (source: model)

'linkedin premium worth it'
  raw model guess: Navigational (0.64) -> after safety rule: Informational (source: model)



In [ ]:
# H3, Part B: do genuinely ambiguous words naturally get LOWER/mixed confidence,
# instead of a falsely confident wrong guess?

ambiguous_queries = ["apple", "amazon", "target", "chrome", "safari", "mercury", "orange"]

print("--- Confidence pattern on genuinely ambiguous words ---")
for q in ambiguous_queries:
    result = classify_query(q, model, vectorizer)
    query_vec = vectorizer.transform([q.lower().strip()])
    probs = model.predict_proba(query_vec)[0]
    print(f"'{q}' -> final: {result['predicted']} (source: {result['source']}, confidence: {result['confidence']:.2f})")
    print(f"   raw model probabilities: Informational={probs[0]:.2f}, Navigational={probs[1]:.2f}, Transactional={probs[2]:.2f}")

--- Confidence pattern on genuinely ambiguous words ---
'apple' -> final: Informational (source: model, confidence: 0.47)
   raw model probabilities: Informational=0.40, Navigational=0.47, Transactional=0.13
'amazon' -> final: Navigational (source: lookup, confidence: 1.00)
   raw model probabilities: Informational=0.36, Navigational=0.53, Transactional=0.11
'target' -> final: Informational (source: model, confidence: 0.53)
   raw model probabilities: Informational=0.53, Navigational=0.41, Transactional=0.07
'chrome' -> final: Informational (source: model, confidence: 0.49)
   raw model probabilities: Informational=0.49, Navigational=0.34, Transactional=0.17
'safari' -> final: Informational (source: model, confidence: 0.59)
   raw model probabilities: Informational=0.59, Navigational=0.34, Transactional=0.06
'mercury' -> final: Informational (source: model, confidence: 0.57)
   raw model probabilities: Informational=0.37, Navigational=0.57, Transactional=0.06
'orange' -> final: Informa